# CORR Data

---

### package imports and basic functions

---

In [1]:
import os
import gc
import sys
import glob
import shutil
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm
from urllib.parse import urlparse
import requests
import zipfile
from pathlib import Path
import polars as pl
# import globus_sdk


In [2]:
from spectranorm import snm

In [3]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


## Extracting data

---

In [28]:
dataset = 'CORR'

In [ ]:
fs_path_dict = {
    row["ID"]: row["path"]
    for _, row in tqdm(pd.read_csv("/home/sina/storage/Normative_Modeling/data/csv/corr/corr_subj.csv").iterrows())
}

len(fs_path_dict), list(fs_path_dict.items())[:1]


In [45]:
data_info_df = pd.read_csv("/home/sina/storage/Normative_Modeling/data/csv/CoRR_AggregatedPhenotypicData.csv")
data_info_df.shape


(4285, 12)

In [46]:
list(data_info_df.columns)


['SUBID',
 'SITE',
 'SESSION',
 'SEX',
 'AGE_AT_SCAN_1',
 'HANDEDNESS',
 'RETEST_DESIGN',
 'RETEST_DURATION',
 'RETEST_UNITS',
 'PRECEDING_CONDITION',
 'VISUAL_STIMULATION_CONDITION',
 'RESTING_STATE_INSTRUCTION']

In [ ]:
data_info_df.head(10)

In [62]:
(data_info_df["SUBID"]==27306).sum()

0

In [48]:
data_info_df[["SEX",]].value_counts()

SEX
#      1741
2      1288
1      1256
Name: count, dtype: int64

In [50]:
data_info_df[["SITE"]].value_counts()

SITE   
NYU_2      489
SWU_4      468
UM         320
HNU_1      300
UPSM_1     231
LMU_2      160
LMU_1      155
BNU_3      144
IPCAS_7    144
BNU_2      122
XHCUMS     121
BMB_1      120
IPCAS_1    120
SWU_1      120
BNU_1      114
MRN_1      108
IPCAS_6     90
MPG_1       88
IPCAS_3     80
Utah_1      78
NYU_1       75
UWM         75
IPCAS_2     68
JHNU_1      60
IACAS       56
IPCAS_5     55
SWU_2       54
LMU_3       50
IBATRT      50
NKI_1       48
SWU_3       46
IPCAS_4     40
IPCAS_8     26
Utah_2      10
Name: count, dtype: int64

In [73]:
data_info_df[["AGE_AT_SCAN_1"]].value_counts(dropna=False)

AGE_AT_SCAN_1
#                1753
23                186
20                180
22                169
24                157
                 ... 
11.32               1
11.34               1
11.38               1
11.5                1
11.56               1
Name: count, Length: 380, dtype: int64

In [ ]:
from contextlib import suppress

sex_encoder = {
    '1': 'F',
    '2': 'M'
}

valid_subjects_dict = {}

for _, row in tqdm(data_info_df.iterrows()):
    key = row["SUBID"]
    subid = f"sub-{key:07d}"
    if subid in fs_path_dict:
        recon_all_path = fs_path_dict[subid]
        if Path(recon_all_path).exists() and row["SESSION"] == "Baseline":
            # with suppress(KeyError): # ignore dictionary misses
            valid_subjects_dict[key] = {
                "unique_id": key,
                "participant_id": key,
                # "session_id": wave,
                "sex": sex_encoder[row["SEX"]],
                "age": float(row["AGE_AT_SCAN_1"]),
                "site": row["SITE"],
                "scan_path": recon_all_path,
                # "diagnosis": (row["DX_GROUP"] == 1),
            }

len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [ ]:
[fs_path_dict[key] for key in fs_path_dict if int(key[4:]) not in valid_subjects_dict]

In [63]:
items = [
    "lh.white", "rh.white",
    "lh.pial", "rh.pial",
    "lh.thickness", "rh.thickness",
    "lh.orig.nofix", "rh.orig.nofix",
    "lh.sphere.reg", "rh.sphere.reg",
]

def directory_is_valid(path, items = items):
    return len([f for f in items if (path / f).exists()]) == len(items)

In [64]:
# Store high-resolution thickness for each individual in a separate file
for idx, subject in enumerate(tqdm(valid_subjects_dict)):
    sub_dir = f"{idx:02d}"[-2:]

    freesurfer_directory = valid_subjects_dict[subject]["scan_path"]
    
    thickness_fslr_output = f"/home/sina/storage/Normative_Modeling/data/fs_LR_32k/{dataset}/{sub_dir}/{subject}.thickness.fslr.npy"

    if (directory_is_valid(Path(freesurfer_directory) / "surf")) and (not Path(thickness_fslr_output).exists()):
        # Compute fslr thickness
        transformed_fslr_thickness = snm.utils.nitools.compute_fslr_thickness(freesurfer_directory)
        np.save(
            ensure_dir(thickness_fslr_output),
            transformed_fslr_thickness.astype(np.float32)
        )


  0%|          | 0/1517 [00:00<?, ?it/s]

In [ ]:
%%time
for idx, subject in enumerate(tqdm(valid_subjects_dict)):
    sub_dir = f"{idx:02d}"[-2:]
    valid_subjects_dict[subject]["subject_index"] = idx
    thickness_fslr_output = f"/home/sina/storage/Normative_Modeling/data/fs_LR_32k/{dataset}/{sub_dir}/{subject}.thickness.fslr.npy"
    if Path(thickness_fslr_output).exists():
        valid_subjects_dict[subject]["thickness"] = np.load(
            thickness_fslr_output,
        ).mean()
    else:
        valid_subjects_dict[subject]["thickness"] = np.nan

len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [77]:
eno_items = [
    "lh.orig.nofix", "rh.orig.nofix",
]

# Compute Euler Number
for idx, subject in enumerate(tqdm(valid_subjects_dict)):
    if "euler_no" not in valid_subjects_dict[subject]:
        sub_dir = f"{idx:02d}"[-2:]
        freesurfer_directory = valid_subjects_dict[subject]["scan_path"]
        if (directory_is_valid(Path(freesurfer_directory) / "surf", items=eno_items)):
            # Compute euler number
            valid_subjects_dict[subject]["euler_no"] = snm.utils.nitools.compute_total_euler_number(
                Path(freesurfer_directory)
            )
        else:
            valid_subjects_dict[subject]["euler_no"] = np.nan


  0%|          | 0/1517 [00:00<?, ?it/s]

In [78]:
# Validity checks
for idx, subject in enumerate(tqdm(valid_subjects_dict)):
    if "validity_check" not in valid_subjects_dict[subject]:
        valid_subjects_dict[subject]["validity_check"] = (
            # (valid_subjects_dict[subject]["diagnosis"] == False)  # Exclude those with a diagnosis
            # and
            (valid_subjects_dict[subject]["thickness"] != np.nan)  # Exclude those missing thickness data
            and
            (valid_subjects_dict[subject]["euler_no"] != np.nan)  # Exclude those missing euler number
        )


  0%|          | 0/1517 [00:00<?, ?it/s]

In [79]:
import joblib

joblib.dump(valid_subjects_dict, ensure_dir(f"/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/subjects.joblib"))


['/home/sina/storage/Normative_Modeling/data/datasets/CORR/subjects.joblib']

In [ ]:
import joblib

# Load the dictionary
valid_subjects_dict = joblib.load(
    f"/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/subjects.joblib"
)

len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [ ]:
final_df = pd.DataFrame({
    'age': [valid_subjects_dict[key]["age"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'thickness': [valid_subjects_dict[key]["thickness"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'sex': [valid_subjects_dict[key]["sex"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'site': [valid_subjects_dict[key]["site"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_ID': [valid_subjects_dict[key]["participant_id"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'euler_no': [valid_subjects_dict[key]["euler_no"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_folder': [valid_subjects_dict[key]["unique_id"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_index': [valid_subjects_dict[key]["subject_index"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
})
final_df['dataset'] = dataset
final_df.head(), final_df.shape


In [83]:
# randomly select only one timepoint per subject (cross-sectional sample)
final_df_subset = final_df.groupby("subject_ID", group_keys=False).sample(n=1, random_state=1234)

# # Only one site:
# final_df_subset.to_parquet(
#     ensure_dir(f'/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/demography.parquet')
# )

# final_df_subset.shape

# Multiple sites:
# Keep only sites with at least 15 subjects
subjects_per_site = final_df_subset.groupby("site")["subject_ID"].nunique()
valid_sites = subjects_per_site[subjects_per_site >= 15].index

final_df_subset[final_df_subset["site"].isin(valid_sites)].to_parquet(
    ensure_dir(f'/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/demography.parquet')
)

final_df_subset[final_df_subset["site"].isin(valid_sites)].shape


(1501, 9)

# ✅ Finished!
